# Fine-tune YOLOX-Tiny for Whiteboard Flowchart Detection

Trains YOLOX-Tiny on the 4-class COCO dataset (`rectangle`, `diamond`, `oval`, `arrowhead`) and exports an ONNX checkpoint + `classes.json` + training config for browser inference.

In [ ]:
!nvidia-smi

In [ ]:
!git clone --depth 1 https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX
%cd /content/YOLOX
!pip install -q -r requirements.txt onnx onnxruntime onnxsim
!pip install -q -e . --no-build-isolation

In [ ]:
# Option A: Generate 3,000 synthetic examples (500 graphs x 6 variants) directly in Colab.
# Upload ml/generate_synthetic_flowcharts.py, ml/yolox_tiny_whiteboard.py, and ml/classes.json to /content/ml/
!python /content/ml/generate_synthetic_flowcharts.py --output-dir /content/dataset --num-graphs 500 --variants-per-graph 6 --seed 42

In [ ]:
# Download pretrained YOLOX-Tiny weights
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth -O /content/yolox_tiny.pth

In [ ]:
import os
os.environ['WHITEBOARD_DATASET_DIR'] = '/content/dataset'

!python /content/YOLOX/tools/train.py \
  -f /content/ml/yolox_tiny_whiteboard.py \
  -d 1 -b 16 --fp16 -o \
  -c /content/yolox_tiny.pth

In [ ]:
!mkdir -p /content/export
!python /content/YOLOX/tools/export_onnx.py \
  --output-name /content/export/yolox_tiny_whiteboard.onnx \
  -f /content/ml/yolox_tiny_whiteboard.py \
  -c /content/YOLOX/YOLOX_outputs/yolox_tiny_whiteboard/best_ckpt.pth \
  --opset 12

!cp /content/ml/classes.json /content/export/classes.json
!cp /content/ml/yolox_tiny_whiteboard.py /content/export/yolox_tiny_whiteboard.py
!cp /content/YOLOX/YOLOX_outputs/yolox_tiny_whiteboard/best_ckpt.pth /content/export/best_ckpt.pth
!ls -lh /content/export